# 15 — Automated Field Delineation (Pampas, Argentina)

Fully automated pipeline requiring **no human-labeled reference boundaries or model training**:

1. Download Google (64-D) and TESSERA (128-D) embeddings → K-means clustering
2. LULC crop filter (Dynamic World) → crop-only polygons per embedding
3. Download Sentinel-2 (10 m) composite
4. SAM2 refines crop polygons from each embedding using S2 imagery
5. 6-way comparison: 2 embeddings × (raw, LULC-filtered, SAM2/S2)

**Study area:** Pampas region near Pergamino, Buenos Aires Province

**Estimated runtime:** ~15–30 minutes (GPU recommended)

**Prerequisites:**
```bash
pip install agribound[gee,samgeo]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
import logging
import warnings
from pathlib import Path

import geopandas as gpd

import agribound

warnings.filterwarnings("ignore", category=FutureWarning, module=r"geedim\..*")
warnings.filterwarnings("ignore", category=RuntimeWarning, module=r"geedim\..*")
warnings.filterwarnings("ignore", message=".*organizePolygons.*")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)
logging.getLogger("googleapiclient").setLevel(logging.CRITICAL)
logging.getLogger("geedim").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.WARNING)

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/pampas_semi_supervised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STUDY_AREA_BBOX = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-60.552058, -33.789222],
                        [-60.274083, -33.734778],
                        [-60.242050, -33.903368],
                        [-60.363950, -33.995060],
                        [-60.505737, -34.015376],
                        [-60.552058, -33.789222],
                    ]
                ],
            },
            "properties": {"name": "Pampas (Pergamino, Buenos Aires)"},
        }
    ],
}

study_area_path = str(OUTPUT_DIR / "study_area.geojson")
with open(study_area_path, "w") as f:
    json.dump(STUDY_AREA_BBOX, f)

YEAR = 2024
MIN_AREA = 5000
CROP_PROB_THRESHOLD = 0.3
SAM_MODEL = "large"
SAM_BATCH_SIZE = 100

## Phase 1: Embedding Clustering (Google + TESSERA)

In [ ]:
# Google Satellite Embeddings (64-D)
google_clusters_path = OUTPUT_DIR / f"all_clusters_google_{YEAR}.gpkg"

if google_clusters_path.exists():
    print(f"Using cached Google clusters: {google_clusters_path}")
    google_clusters_gdf = gpd.read_file(google_clusters_path)
else:
    print(f"Downloading Google embeddings and clustering ({YEAR})...")
    google_clusters_gdf = agribound.delineate(
        study_area=study_area_path,
        source="google-embedding",
        year=YEAR,
        engine="embedding",
        output_path=str(google_clusters_path),
        gee_project=GEE_PROJECT,
        device="cpu",
        min_area=MIN_AREA,
        lulc_filter=False,
        engine_params={},
    )

print(f"Google clusters: {len(google_clusters_gdf)} polygons")

# TESSERA Embeddings (128-D)
tessera_clusters_path = OUTPUT_DIR / f"all_clusters_tessera_{YEAR}.gpkg"

if tessera_clusters_path.exists():
    print(f"Using cached TESSERA clusters: {tessera_clusters_path}")
    tessera_clusters_gdf = gpd.read_file(tessera_clusters_path)
else:
    print(f"Downloading TESSERA embeddings and clustering ({YEAR})...")
    tessera_clusters_gdf = agribound.delineate(
        study_area=study_area_path,
        source="tessera-embedding",
        year=YEAR,
        engine="embedding",
        output_path=str(tessera_clusters_path),
        gee_project=GEE_PROJECT,
        device="cpu",
        min_area=MIN_AREA,
        lulc_filter=False,
        engine_params={},
    )

print(f"TESSERA clusters: {len(tessera_clusters_gdf)} polygons")

## Phase 2: LULC Crop Filter → Crop-Only Polygons

In [ ]:
from agribound.config import AgriboundConfig
from agribound.postprocess.lulc_filter import filter_by_lulc


def _lulc_filter(clusters_gdf, source_name, cache_name):
    """LULC-filter clusters, with caching."""
    path = OUTPUT_DIR / f"fields_crop_{cache_name}_{YEAR}.gpkg"
    if path.exists():
        print(f"Using cached {cache_name} crop polygons: {path}")
        return gpd.read_file(path)
    config = AgriboundConfig(
        study_area=study_area_path,
        source=source_name,
        year=YEAR,
        output_path=str(path),
        gee_project=GEE_PROJECT,
        lulc_crop_threshold=CROP_PROB_THRESHOLD,
    )
    result = filter_by_lulc(clusters_gdf, config)
    result.to_file(path, driver="GPKG", layer="fields")
    return result


google_crop_gdf = _lulc_filter(google_clusters_gdf, "google-embedding", "google")
print(f"Google crop polygons: {len(google_crop_gdf)} (from {len(google_clusters_gdf)})")

tessera_crop_gdf = _lulc_filter(tessera_clusters_gdf, "tessera-embedding", "tessera")
print(f"TESSERA crop polygons: {len(tessera_crop_gdf)} (from {len(tessera_clusters_gdf)})")

## Phase 3: Download Sentinel-2 Composite

In [ ]:
import pandas as pd

from agribound.composites import get_composite_builder

all_crop_gdf = pd.concat([google_crop_gdf, tessera_crop_gdf], ignore_index=True)
ref_bounds_gdf = all_crop_gdf.to_crs(epsg=4326)
bounds = ref_bounds_gdf.total_bounds
composite_study_area_path = str(OUTPUT_DIR / "study_area_composite.geojson")
composite_bbox = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [bounds[0], bounds[1]],
                        [bounds[2], bounds[1]],
                        [bounds[2], bounds[3]],
                        [bounds[0], bounds[3]],
                        [bounds[0], bounds[1]],
                    ]
                ],
            },
            "properties": {"name": "Composite extent"},
        }
    ],
}
with open(composite_study_area_path, "w") as f:
    json.dump(composite_bbox, f)

s2_config = AgriboundConfig(
    study_area=composite_study_area_path,
    source="sentinel2",
    year=YEAR,
    output_path=str(OUTPUT_DIR / "s2_composite.gpkg"),
    gee_project=GEE_PROJECT,
    composite_method="median",
    cloud_cover_max=20,
    date_range=(f"{YEAR}-10-01", f"{YEAR}-10-31"),
)
s2_raster = get_composite_builder("sentinel2").build(s2_config)
print(f"Sentinel-2 composite: {s2_raster}")

## Phase 4: SAM2 Boundary Refinement (Google + TESSERA on S2)

In [ ]:
from agribound.engines.samgeo_engine import refine_boundaries
from agribound.postprocess.simplify import simplify_polygons, smooth_polygons


def _run_sam2(crop_gdf, label, raster_path, out_path):
    """Run SAM2 refinement for a single embedding set."""
    if out_path.exists():
        print(f"Using cached {out_path.name}")
        return gpd.read_file(out_path)
    try:
        print(f"Refining {len(crop_gdf)} {label} polygons with SAM2...")
        sam_config = AgriboundConfig(
            source="sentinel2",
            engine="embedding",
            year=YEAR,
            study_area=composite_study_area_path,
            output_path=str(out_path),
            engine_params={
                "sam_model": SAM_MODEL,
                "sam_batch_size": SAM_BATCH_SIZE,
            },
            device="auto",
        )
        result = refine_boundaries(crop_gdf, raster_path, sam_config)
        result = smooth_polygons(result, iterations=3)
        result = simplify_polygons(result, tolerance=2.0)
        result.to_file(out_path, driver="GPKG", layer="fields")
        print(f"→ {len(result)} fields")
        return result
    except Exception as exc:
        print(f"SAM2 failed: {exc}")
        return None


# Google crops + S2
google_s2_gdf = _run_sam2(
    google_crop_gdf,
    "Google",
    s2_raster,
    OUTPUT_DIR / f"fields_sam2_google_s2_{YEAR}.gpkg",
)

# TESSERA crops + S2
tessera_s2_gdf = _run_sam2(
    tessera_crop_gdf,
    "TESSERA",
    s2_raster,
    OUTPUT_DIR / f"fields_sam2_tessera_s2_{YEAR}.gpkg",
)

## Phase 5: Comparison

In [ ]:
comparison_items = [
    ("Google clusters (raw)", google_clusters_gdf),
    ("Google crops (LULC)", google_crop_gdf),
    ("TESSERA clusters (raw)", tessera_clusters_gdf),
    ("TESSERA crops (LULC)", tessera_crop_gdf),
]
if google_s2_gdf is not None:
    comparison_items.append(("Google + SAM2/S2 (10 m)", google_s2_gdf))
if tessera_s2_gdf is not None:
    comparison_items.append(("TESSERA + SAM2/S2 (10 m)", tessera_s2_gdf))

print(f"{'Method':<35} {'Fields':>8} {'Area (ha)':>12}")
print(f"{'-' * 35} {'-' * 8} {'-' * 12}")

for label, gdf in comparison_items:
    area = gdf["metrics:area"].sum() / 10000 if "metrics:area" in gdf.columns else 0
    print(f"{label:<35} {len(gdf):>8} {area:>12,.1f}")

## Visualization

In [ ]:
from agribound.visualize import show_comparison

comp_gdfs = [item[1] for item in comparison_items]
comp_labels = [item[0] for item in comparison_items]

m = show_comparison(
    comp_gdfs,
    labels=comp_labels,
    basemap="Esri.WorldImagery",
    output_html=str(OUTPUT_DIR / "map_comparison.html"),
)
m